In [32]:
from pathlib import Path
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_DIR = Path("/home/sahil-narkhede/Desktop/Me/SEM6/cv/OncoPRISM/data")
LIMIT = 5000
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [ ]:
!pip install -q h5py timm torch torchvision torch-geometric networkx scikit-learn

In [33]:
import h5py

def load_subset(x_path, y_path, limit=5000):
    with h5py.File(x_path, "r") as f:
        X = f["x"][:limit]

    with h5py.File(y_path, "r") as f:
        y = f["y"][:limit]

    return X, y.squeeze()

In [34]:
X, y = load_subset(
    DATA_DIR / "camelyonpatch_level_2_split_train_x.h5",
    DATA_DIR / "camelyonpatch_level_2_split_train_y.h5",
    limit=LIMIT,
 )

print("Loaded train subset:", X.shape, y.shape)

Loaded train subset: (5000, 96, 96, 3) (5000,)


In [ ]:
import h5py

with h5py.File(DATA_DIR / "camelyonpatch_level_2_split_train_x.h5", "r") as f:
    print("Available datasets:", list(f.keys()))

['x']


In [8]:
print(X.shape)

(5000, 96, 96, 3)


PREPROCESSING


In [35]:
import cv2
import numpy as np

def preprocess(X, size=(128, 128)):
    out = [cv2.resize(img, size) for img in X]
    return np.asarray(out, dtype=np.float32) / 255.0

X = preprocess(X)
print("Preprocessed:", X.shape, X.dtype, X.min(), X.max())

Preprocessed: (5000, 128, 128, 3) float32 0.0 1.0


In [36]:
import torch
from torch.utils.data import Dataset, DataLoader

class PCamDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        img = torch.tensor(self.X[idx], dtype=torch.float32).permute(2, 0, 1)
        label = torch.tensor(self.y[idx], dtype=torch.long)
        return img, label

dataset = PCamDataset(X, y)
loader = DataLoader(dataset, batch_size=32, shuffle=False)

In [37]:
import timm
import torch.nn as nn

model = timm.create_model("resnet18", pretrained=True)
model.fc = nn.Identity()
model.to(device)
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (act1): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (drop_block): Identity()
      (act1): ReLU(inplace=True)
      (aa): Identity()
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act2): ReLU(inplace=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, m

In [38]:
features = []
labels = []

with torch.no_grad():
    for imgs, y_batch in loader:
        imgs = imgs.to(device)
        feats = model(imgs)
        features.append(feats.cpu())
        labels.append(y_batch)

features = torch.cat(features)
labels = torch.cat(labels).long()

print("Feature matrix:", features.shape, "Labels:", labels.shape)

Feature matrix: torch.Size([5000, 512]) Labels: torch.Size([5000])


In [39]:
from sklearn.neighbors import NearestNeighbors

def build_graph(features, k=8):
    nbrs = NearestNeighbors(n_neighbors=k + 1, metric="euclidean")
    nbrs.fit(features)
    _, indices = nbrs.kneighbors(features)

    edge_pairs = set()
    for i in range(len(indices)):
        for j in indices[i][1:]:  # skip self-neighbor
            edge_pairs.add((i, int(j)))
            edge_pairs.add((int(j), i))

    return sorted(edge_pairs)

In [40]:
edge_index = build_graph(features.numpy(), k=8)
print("Number of directed edges:", len(edge_index))

Number of directed edges: 67714


In [ ]:
# torch-geometric is installed above; this cell is intentionally left as a no-op.

In [41]:
import torch
from torch_geometric.data import Data

edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

x = features
y = labels

data = Data(x=x, edge_index=edge_index, y=y)

print(data)

Data(x=[5000, 512], edge_index=[2, 67714], y=[5000])


In [ ]:
import networkx as nx

def compute_pagerank(edge_index, num_nodes, alpha=0.85):
    G = nx.DiGraph()
    G.add_nodes_from(range(num_nodes))
    G.add_edges_from(edge_index.t().tolist())

    pr = nx.pagerank(G, alpha=alpha)
    scores = torch.tensor([pr[i] for i in range(num_nodes)], dtype=torch.float32)
    # scores = scores / (scores.max() + 1e-8)
    scores = torch.log(scores + 1e-8)
    scores = (scores - scores.mean()) / scores.std()
    return scores

In [43]:
pagerank_scores = compute_pagerank(edge_index, num_nodes=features.shape[0])
print("PageRank shape:", pagerank_scores.shape, "min/max:", pagerank_scores.min().item(), pagerank_scores.max().item())

PageRank shape: torch.Size([5000]) min/max: 0.1559017449617386 0.9999867081642151


In [44]:
x_aug = torch.cat([x, pagerank_scores.view(-1, 1)], dim=1)

print("Augmented features:", x_aug.shape)

Augmented features: torch.Size([5000, 513])


In [45]:
from torch_geometric.nn import GCNConv
import torch.nn.functional as F

class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(513, 128)
        self.conv2 = GCNConv(128, 2)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

In [46]:
from sklearn.model_selection import train_test_split

model = GCN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

data = data.to(device)
data.x = x_aug.to(device)
data.y = data.y.view(-1).long()

num_nodes = data.num_nodes
all_idx = np.arange(num_nodes)
train_idx, temp_idx = train_test_split(all_idx, test_size=0.3, random_state=SEED, stratify=data.y.cpu().numpy())
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=SEED, stratify=data.y.cpu().numpy()[temp_idx])

train_mask = torch.zeros(num_nodes, dtype=torch.bool, device=device)
val_mask = torch.zeros(num_nodes, dtype=torch.bool, device=device)
test_mask = torch.zeros(num_nodes, dtype=torch.bool, device=device)
train_mask[torch.tensor(train_idx, device=device)] = True
val_mask[torch.tensor(val_idx, device=device)] = True
test_mask[torch.tensor(test_idx, device=device)] = True

for epoch in range(1, 21):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[train_mask], data.y[train_mask])
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(data.x, data.edge_index)
        val_loss = F.cross_entropy(logits[val_mask], data.y[val_mask])
    print(f"Epoch {epoch:02d} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")

Epoch 01 | Train Loss: 0.6969 | Val Loss: 2.3376
Epoch 02 | Train Loss: 2.3289 | Val Loss: 1.1651
Epoch 03 | Train Loss: 1.1092 | Val Loss: 1.0395
Epoch 04 | Train Loss: 0.9874 | Val Loss: 0.5857
Epoch 05 | Train Loss: 0.5547 | Val Loss: 0.6999
Epoch 06 | Train Loss: 0.6812 | Val Loss: 0.7424
Epoch 07 | Train Loss: 0.7268 | Val Loss: 0.5946
Epoch 08 | Train Loss: 0.5776 | Val Loss: 0.5130
Epoch 09 | Train Loss: 0.4913 | Val Loss: 0.5801
Epoch 10 | Train Loss: 0.5552 | Val Loss: 0.6156
Epoch 11 | Train Loss: 0.5908 | Val Loss: 0.5655
Epoch 12 | Train Loss: 0.5427 | Val Loss: 0.5076
Epoch 13 | Train Loss: 0.4881 | Val Loss: 0.5048
Epoch 14 | Train Loss: 0.4889 | Val Loss: 0.5343
Epoch 15 | Train Loss: 0.5202 | Val Loss: 0.5371
Epoch 16 | Train Loss: 0.5230 | Val Loss: 0.5071
Epoch 17 | Train Loss: 0.4915 | Val Loss: 0.4884
Epoch 18 | Train Loss: 0.4695 | Val Loss: 0.5097
Epoch 19 | Train Loss: 0.4866 | Val Loss: 0.5159
Epoch 20 | Train Loss: 0.4917 | Val Loss: 0.4950


In [47]:
model.eval()

with torch.no_grad():
    out = model(data.x, data.edge_index)
    preds = out.argmax(dim=1)

In [24]:
out = model(data.x, data.edge_index)

In [48]:
y_true = data.y[test_mask].cpu().numpy()
y_pred = preds[test_mask].cpu().numpy()

In [49]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print("Test Accuracy:", acc)
print("Test Precision:", prec)
print("Test Recall:", rec)
print("Test F1 Score:", f1)

Test Accuracy: 0.7826666666666666
Test Precision: 0.8
Test Recall: 0.7643979057591623
Test F1 Score: 0.7817938420348058


---

## PR-GAT

In [ ]:
def normalize_pagerank(pr):
    pr = torch.log(pr + 1e-8)
    pr = (pr - pr.mean()) / (pr.std() + 1e-8)
    return pr

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.nn import MessagePassing

class PRGAT(torch.nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.gat1 = GATConv(in_channels, 128, heads=4, concat=True)
        self.gat2 = GATConv(128 * 4, 2, heads=1, concat=False)

    def forward(self, x, edge_index, pagerank):
        # Reweight each node embedding by scalar PageRank score.
        x = x * pagerank.view(-1, 1)
        x = self.gat1(x, edge_index)
        x = F.relu(x)
        x = self.gat2(x, edge_index)
        return x
    
class PRGATNet(nn.Module):
    def __init__(self, in_channels):
        super().__init__()

        self.conv1 = PRGATConv(in_channels, 64, heads=4)
        self.conv2 = PRGATConv(64 * 4, 32, heads=4)
        self.conv3 = PRGATConv(32 * 4, 2, heads=1)

        self.dropout = 0.3

    def forward(self, x, edge_index, pagerank):
        x = self.conv1(x, edge_index, pagerank)
        x = F.elu(x)
        x = x.view(x.size(0), -1)

        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index, pagerank)
        x = F.elu(x)
        x = x.view(x.size(0), -1)

        x = self.conv3(x, edge_index, pagerank)

        return x

class PRGATLayer(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')
        self.W = torch.nn.Linear(in_channels, out_channels)
        self.att = torch.nn.Parameter(torch.randn(2 * out_channels))

    def forward(self, x, edge_index, pagerank):
        x = self.W(x)
        return self.propagate(edge_index, x=x, pagerank=pagerank)

    def message(self, x_i, x_j, pagerank_j):
        # attention score
        alpha = torch.cat([x_i, x_j], dim=-1)
        alpha = (alpha * self.att).sum(dim=-1)

        # integrate PageRank here (KEY STEP)
        alpha = alpha * pagerank_j

        alpha = F.leaky_relu(alpha)
        alpha = torch.softmax(alpha, dim=0)

        return x_j * alpha.unsqueeze(-1)

In [51]:
model = PRGAT(data.x.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

data = data.to(device)
pagerank_scores = pagerank_scores.to(device).view(-1)
data.y = data.y.view(-1).long()

for epoch in range(1, 21):
    model.train()
    optimizer.zero_grad()

    out = model(data.x, data.edge_index, pagerank_scores)
    loss = F.cross_entropy(out[train_mask], data.y[train_mask])

    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_out = model(data.x, data.edge_index, pagerank_scores)
        val_loss = F.cross_entropy(val_out[val_mask], data.y[val_mask])
    print(f"PRGAT Epoch {epoch:02d} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")

model.eval()
with torch.no_grad():
    test_out = model(data.x, data.edge_index, pagerank_scores)
    test_preds = test_out.argmax(dim=1)

y_true_prgat = data.y[test_mask].cpu().numpy()
y_pred_prgat = test_preds[test_mask].cpu().numpy()

print("PRGAT Test Accuracy:", accuracy_score(y_true_prgat, y_pred_prgat))
print("PRGAT Test Precision:", precision_score(y_true_prgat, y_pred_prgat, zero_division=0))
print("PRGAT Test Recall:", recall_score(y_true_prgat, y_pred_prgat, zero_division=0))
print("PRGAT Test F1:", f1_score(y_true_prgat, y_pred_prgat, zero_division=0))

PRGAT Epoch 01 | Train Loss: 0.6938 | Val Loss: 0.7936
PRGAT Epoch 02 | Train Loss: 0.7838 | Val Loss: 0.6555
PRGAT Epoch 03 | Train Loss: 0.6378 | Val Loss: 0.6673
PRGAT Epoch 04 | Train Loss: 0.6485 | Val Loss: 0.5748
PRGAT Epoch 05 | Train Loss: 0.5586 | Val Loss: 0.5721
PRGAT Epoch 06 | Train Loss: 0.5585 | Val Loss: 0.5802
PRGAT Epoch 07 | Train Loss: 0.5666 | Val Loss: 0.5481
PRGAT Epoch 08 | Train Loss: 0.5323 | Val Loss: 0.5264
PRGAT Epoch 09 | Train Loss: 0.5072 | Val Loss: 0.5382
PRGAT Epoch 10 | Train Loss: 0.5159 | Val Loss: 0.5336
PRGAT Epoch 11 | Train Loss: 0.5097 | Val Loss: 0.5108
PRGAT Epoch 12 | Train Loss: 0.4866 | Val Loss: 0.5042
PRGAT Epoch 13 | Train Loss: 0.4803 | Val Loss: 0.5106
PRGAT Epoch 14 | Train Loss: 0.4863 | Val Loss: 0.5050
PRGAT Epoch 15 | Train Loss: 0.4791 | Val Loss: 0.4970
PRGAT Epoch 16 | Train Loss: 0.4686 | Val Loss: 0.5039
PRGAT Epoch 17 | Train Loss: 0.4728 | Val Loss: 0.5027
PRGAT Epoch 18 | Train Loss: 0.4714 | Val Loss: 0.4909
PRGAT Epoc